In [2]:
import sage.all as sage
print("Sage version:", sage.version())


Sage version: SageMath version 10.8, Release Date: 2025-12-18


In [3]:
from sage.all import*

In [4]:
def Pivot(A, i, W):
    m = A.ncols()
    I = -1
    D = -1
    DW = -1  # shifted pivot degree

    for j in range(m):
        if A[i,j] != 0:
            if A[i,j].degree() + W[j] >= DW: 
                DW = A[i,j].degree()+W[j]
                D = A[i,j].degree()
                I = j

    return (I, D, DW)

# Example

F = GF(7)
R = PolynomialRing(F, "x")
x = R.gen()

A = Matrix(R, [
    [0,      0,    0],
    [2*x**3 + x**2,   4*x**2,      x + 6],
    [0,               x**3,        2*x**2 + 5],
])

i = 0

W = [0,0,0]

Pivot(A,i,W)

(-1, -1, -1)

In [5]:
def SimpleTransformation(A,i,j,I,d_i,d_j):
    m = A.ncols()
    d = d_i-d_j

    cxe = (A[i,I].leading_coefficient() / A[j,I].leading_coefficient())*(x**d)
    for l in range(m):
        A[i,l] = A[i,l] - cxe*A[j,l]

    return A

# Example

F = GF(7)
R = PolynomialRing(F, "x")
x = R.gen()

A5 = Matrix(R, [
    [x**2 + 2*x + 1,   3*x + 2,      4],
    [2*x**2 + 4*x + 2, 6*x + 4,      1], 
    [x + 3,           0,            5],
])

i = 0
j = 1
I = 1
d_i = 1
d_j = 1


SimpleTransformation(A5,i,j,I, d_i,d_j)

[              0               0               0]
[2*x^2 + 4*x + 2         6*x + 4               1]
[          x + 3               0               5]

In [6]:
def WeakPopov(A,W=0):
    n = A.nrows()
    # If no weights are given, set W = [0,0,...,0]
    if W == 0:
        W=zero_vector(ZZ, A.nrows())
        
    pivots = zero_matrix(ZZ, n, 2)

    control = 1

    for i in range(n):
        pivots[i,0] = Pivot(A,i,W)[0] 
        pivots[i,1] = Pivot(A,i,W)[1]

    while control == 1:

        control = 0
        broke = False

        for i in range(n):
            for j in range(i+1,n):
                if pivots[i,0] == pivots[j,0]:
                    if pivots[i,1] >= pivots[j,1]:
                        A = SimpleTransformation(A,i,j,pivots[i,0],pivots[i,1],pivots[j,1])
                        pivots[i,0] = Pivot(A,i,W)[0]
                        pivots[i,1] = Pivot(A,i,W)[1]
                    else:
                        A = SimpleTransformation(A,j,i,pivots[j,0],pivots[j,1],pivots[i,1])
                        pivots[j,0] = Pivot(A,j,W)[0]
                        pivots[j,1] = Pivot(A,j,W)[1]
                    control = 1
                    broke = True
                    break
            if broke:
                break
                        

    return A
    

# Example

F = GF(7)
R = PolynomialRing(F, "x")
x = R.gen()

A = Matrix(R, [
    [4*x**2 + 3*x + 5,      4*x**2 + 3*x + 4,    6*x**2 + 1],
    [3*x + 6,   3*x + 5,      3 + x],
    [6*x**2 + 4*x + 2,               6*x**2,        2*x**2 + x],
])

A5 = Matrix(R, [
    [x**2 + 2*x + 1,   3*x + 2,      4],
    [2*x**2 + 4*x + 2, 6*x + 4,      1],  # 2 * row0 in col0/col1; may cancel heavily
    [x + 3,           0,            5],
])

W = [0,0,0]
WeakPopov(A5,W)

[      0       0       0]
[5*x + 2 6*x + 4 4*x + 1]
[  x + 3       0       5]

In [8]:
[      0       0       0]
[5*x + 2 6*x + 4 4*x + 1]
[  x + 3       0       5]

SyntaxError: invalid syntax. Perhaps you forgot a comma? (536596398.py, line 1)

In [7]:
def MakeMonic(A,pivots,n,m):

    for i in range(n):
        if A[i,pivots[i,0]].leading_coefficient() != 1 and A[i,pivots[i,0]].leading_coefficient() != 0:
            monic_const = 1 / A[i,pivots[i,0]].leading_coefficient() 
            for j in range(m):
                A[i,j] = monic_const*A[i,j]  
    return A   

In [8]:
def MakeYMonic(A, n, m):
    last_col = m - 1

    for i in range(n):
        y = A[i, last_col]
        if y != 0 and y.leading_coefficient() != 1:
            monic_const = 1 / y.leading_coefficient()
            for j in range(m):
                A[i, j] = monic_const * A[i, j]
    return A

In [9]:
def AscendingOrder(A, W):
    n = A.nrows()
    check = 1
    while check == 1:
        check = 0
        for i in range(n-1):
            I1, D1, DW1 = Pivot(A, i, W)
            I2, D2, DW2 = Pivot(A, i+1, W)

            if DW1 > DW2:
                A.swap_rows(i, i+1)
                check = 1
            elif DW1 == DW2:
                if I1 > I2:
                    A.swap_rows(i, i+1)
                    check = 1
    return A

In [10]:
AscendingOrder(WeakPopov(A5,W),W)

[      0       0       0]
[  x + 3       0       5]
[5*x + 2 6*x + 4 4*x + 1]

In [11]:
def PopovForm(A,W=0):
    n = A.nrows()
    # If no weights are given, set W = [0,0,...,0]
    if W == 0:
        W=zero_vector(ZZ, A.nrows())
    m = A.ncols()
    pivots = zero_matrix(ZZ, n, 2)
    C = []

    A = WeakPopov(A,W)
    A = AscendingOrder(A,W)

    for i in range(n):
        pivots[i,0] = Pivot(A,i,W)[0]
        pivots[i,1] = Pivot(A,i,W)[1]
    
    for k in range(n):
        if not A[k].is_zero():
            C.append(k)
            for i in C:
                if i<k:
                    delta = A[k,pivots[i,0]].degree() - pivots[i,1]
                    if delta >= 0:
                        A = SimpleTransformation(A,k,i,pivots[i,0],A[k,pivots[i,0]].degree(),pivots[i,1])
    A = MakeMonic(A,pivots,n,m)  
    #A = MakeYMonic(A,n,m)    
    return A
            
F = GF(7)
R = PolynomialRing(F, "x")
x = R.gen()

A1 = Matrix(R, [
    [x**2 + 1,      x**2 + 2,      0],
    [x**2 + 3,      0,            x**2 + 4],
    [0,            x + 1,        1],
])

W = [1,2,3]
PopovForm(A1)

[          0       x + 1           1]
[    x^2 + 1           3     6*x + 1]
[          2           4 x^2 + x + 3]

In [12]:
def G_Poly(eval,R):
    x = R.gen()
    G = 1
    for xi in eval:
        G *= (x-xi)
    return G

In [13]:
def LagrangePolynomials(eval, R, n):
    x = R.gen()
    L = vector(R, [1]*n)
    for i in range(n):
        for l in range(n):
            if l == i:
                continue
            L[i] *= (x-eval[l])/(eval[i]-eval[l])
    return L

In [14]:
def R_Poly(L,r,n):
    RX = 0
    for i in range(n):
        RX += r[i]*L[i]

    return RX

In [15]:
F = GF(13)
R = PolynomialRing(F, "x")
x = R.gen()
eval = vector(F, [1,2,3,4])
r = vector(F, [2,3,4,8])
n = 4

G = G_Poly(eval,R)
L = LagrangePolynomials(eval, R, n)
RX = R_Poly(L,r,n)

M = Matrix(R, [
    [G,      0,      0],
    [-RX,      1,      0],
    [-RX**2,   0,      1],
])

W = [0,1,2]
PopovForm(M,W)

[  12*x^2 + 3*x + 4              x + 9                  0]
[12*x^2 + 11*x + 12                  0                  1]
[   x^3 + 7*x^2 + 9                 11                  0]

In [16]:
F = GF(13)
R = PolynomialRing(F, "x,y")

x, y = R.gens()

Q = x^2 + 2*x + 1 + 12*y^2
Q.factor()

(-1) * (x + y + 1) * (-x + y - 1)

In [26]:
def SudanListDecoding(eval,r,l,R,k):
    n = eval.length()
    G = G_Poly(eval,R)
    L = LagrangePolynomials(eval, R, n)
    RX = R_Poly(L,r,n)
    M = matrix.identity(R, l+1, l+1)
    M[0,0] = G
    for i in range(1,l+1):
        M[i,0] = -RX**i
    W = vector(ZZ, range(l+2))
    M = PopovForm(M,W)
    # Choose row with lowest weightet degree
    best_row = None
    best_degree = None
    for i in range(l+1):
        rowdegree = None
        for j in range(l+1):
            if rowdegree is None:
                rowdegree = M[i,j].degree()+W[j]
            else:
                rowdegree = max(M[i,j].degree()+W[j], rowdegree)
        if best_degree is None or best_degree >= rowdegree:
            best_row = i
            best_degree = rowdegree
    Q = 0
    for i in range(l+1):
        Q += M[best_row][i]*y**i
    print(Q)
    
    # Check that f agrees with enough points

    candidates = []
    valid_candidates = []
    tau = floor(n*l/(l+1)-l/2*(k-1))
    threshold = n - tau

    for factor, _ in Q.factor():
        if factor.degree(y) == 1:
            a = factor.coefficient(y) 
            b = factor - a*y 

            f = -b / a      
            candidates.append(f)

            agreements = 0
            for xi, ri in zip(eval, r):
                if f(xi,y) == ri:
                    agreements += 1
            if agreements >= threshold:
                valid_candidates.append(f)
    return candidates, valid_candidates           

k = 2
l = 2

F = GF(13)
R = PolynomialRing(F, "x,y")

x, y = R.gens()

eval = vector(F, [1,2,3,4,5,6,7,8])
r = vector(F, [2,3,4,5,1,2,3,4])

SudanListDecoding(eval, r, l, R, k)

x^2 - 2*x*y + y^2 - 3*x + 3*y - 4


([x - 4, x + 1], [x - 4, x + 1])

In [18]:
k = 2
l = 3

F = GF(13)
R = PolynomialRing(F, "x,y")

x, y = R.gens()

eval = vector(F, [1,2,3,4,5,6])
r = vector(F, [2,3,4,8,6,3])
n = 6

#floor(n*l/(l+1)-l/2*(k-1))
SudanListDecoding(eval, r, l, R, k)

4*x^2 - 5*x*y + y^2 - 4*x - 6*y + 5


([4*x + 5, x + 1], [4*x + 5, x + 1])

In [22]:
def GurusamiSudanListDecoding(eval,r,l,R,k,m=1):   # CHANGED: added multiplicity parameter m
    n = eval.length()
    G = G_Poly(eval,R)
    L = LagrangePolynomials(eval, R, n)
    RX = R_Poly(L,r,n)

    M = zero_matrix(R, l+1, l+1)   # CHANGED: build Lee-O'Sullivan basis

    for i in range(l+1):
        u = max(i - m, 0)          # CHANGED: [i-s] = max(i-m,0)
        P = G**u * (y - RX)**(i-u) * y**u   # CHANGED: standard basis

        for j in range(l+1):
            M[i,j] = P.coefficient(y**j)

    # CHANGED: weighted degree should be x-degree + j*(k-1)
    W = vector(ZZ, [j*(k-1) for j in range(l+1)])
    M = WeakPopov(M,W)

    # Choose row with lowest weighted degree
    best_row = None
    best_degree = None
    for i in range(l+1):
        rowdegree = None
        for j in range(l+1):
            if rowdegree is None:
                rowdegree = M[i,j].degree()+W[j]
            else:
                rowdegree = max(M[i,j].degree()+W[j], rowdegree)
        if best_degree is None or best_degree >= rowdegree:
            best_row = i
            best_degree = rowdegree

    Q = 0
    for i in range(l+1):
        Q += M[best_row][i]*y**i
    print(Q)
    print(Q.factor())
    
    # Check that f agrees with enough points

    candidates = []
    valid_candidates = []

    # CHANGED: multiplicity condition is m * agreements > weighted degree of Q
    tau = floor(n*l/(l+1)-l/2*(k-1))
    threshold = (n - tau)
    #threshold = floor(best_degree / m) + 1

    for factor, _ in Q.factor():
        if factor.degree(y) == 1:
            a = factor.coefficient(y) 
            b = factor - a*y 

            f = -b / a      
            candidates.append(f)

            agreements = 0
            for xi, ri in zip(eval, r):
                if f(xi,y) == ri:
                    agreements += 1
            if agreements >= threshold:
                valid_candidates.append(f)

    return candidates, valid_candidates

In [26]:
k = 2
l = 3

F = GF(13)
R = PolynomialRing(F, "x,y")

x, y = R.gens()

eval = vector(F, [1,2,3,4,5,6])
r = vector(F, [2,3,4,8,6,3])
n = 6

#floor(n*l/(l+1)-l/2*(k-1))
SudanListDecoding(eval, r, l, R, k)

4*x^2 - 5*x*y + y^2 - 4*x - 6*y + 5


([4*x + 5, x + 1], [4*x + 5, x + 1])

In [24]:
k = 2
l = 2

F = GF(13)
R = PolynomialRing(F, "x,y")

x, y = R.gens()

eval = vector(F, [1,2,3,4])
r = vector(F, [2,3,4,8])

GurusamiSudanListDecoding(eval, r, l, R, k,1)

2*y + 2
(2) * (y + 1)


([-1], [])

In [28]:
def SudanListDecodingchat(eval,r,l,q,k,m=1):
    n = eval.length()

    F = GF(q)
    R = PolynomialRing(F, "x,y")

    # CHANGED: make a univariate ring S = F[x] for the Popov matrix
    F = R.base_ring()
    S = PolynomialRing(F, "x")
    xS = S.gen()

    # CHANGED: keep bivariate variables only for final Q and factorization
    x, y = R.gens()

    # CHANGED: compute G and RX in S, not in R = F[x,y]
    G = prod(xS - S(xi) for xi in eval)
    L = LagrangePolynomials(eval, S, n)
    RX = R_Poly(L, r, n)

    # CHANGED: build matrix over S = F[x]
    M = zero_matrix(S, l+1, l+1)

    # CHANGED: standard multiplicity basis:
    # for i <= m:  G^(m-i)(y-RX)^i
    # for i >  m:  y^(i-m)(y-RX)^m
    for i in range(l+1):
        if i <= m:
            for j in range(i+1):
                M[i,j] = binomial(i,j) * (-RX)**(i-j) * G**(m-i)
        else:
            for j in range(m+1):
                M[i,j+i-m] = binomial(m,j) * (-RX)**(m-j)

    # CHANGED: weighted degree should be x-degree + j*(k-1)
    W = vector(ZZ, [j*(k-1) for j in range(l+1)])

    M = PopovForm(M,W)

    # Choose row with lowest weighted degree
    best_row = None
    best_degree = None
    for i in range(l+1):
        rowdegree = None
        for j in range(l+1):
            if M[i,j] == 0:       # CHANGED: skip zero entries
                continue

            deg = M[i,j].degree() + W[j]

            if rowdegree is None:
                rowdegree = deg
            else:
                rowdegree = max(deg, rowdegree)

        if rowdegree is not None and (best_degree is None or best_degree >= rowdegree):
            best_row = i
            best_degree = rowdegree

    # CHANGED: reconstruct Q in the bivariate ring R
    Q = 0
    for i in range(l+1):
        Q += R(M[best_row][i]) * y**i

    print(Q)

    candidates = []
    valid_candidates = []

    # CHANGED: multiplicity decoding condition
    threshold = floor(best_degree/m) + 1

    for factor, _ in Q.factor():
        if factor.degree(y) == 1:
            a = factor.coefficient(y)
            b = factor - a*y

            f = -b / a
            candidates.append(f)

            agreements = 0
            for xi, ri in zip(eval, r):
                if f(xi, y) == ri:
                    agreements += 1

            if agreements >= threshold:
                valid_candidates.append(f)

    return candidates, valid_candidates

k = 2
l = 2
q = 13
F = GF(q)
eval = vector(F, [1,2,3,4])
r = vector(F, [2,3,4,8])

SudanListDecodingchat(eval, r, l, q, k,4)

x^12 - 2*x^11*y + x^10*y^2 + 2*x^10*y - 2*x^9*y^2 + 5*x^10 + x^9*y - 5*x^8*y^2 + 4*x^9 + 4*x^8*y + 3*x^7*y^2 + 6*x^8 - 3*x^7*y + 5*x^6*y^2 + 6*x^7 + 4*x^6*y + 6*x^5*y^2 - 5*x^6 + 6*x^5*y + 4*x^4*y^2 - 4*x^5 + 2*x^4*y - 5*x^3*y^2 + 5*x^4 + x^3*y - 2*x^2*y^2 - x^2*y - 4*x*y^2 + 2*x^2 - 3*x*y - y^2 - 6*x + 2*y - 1


([x + 1], [])

In [1]:
k = 2
l = 3
q=13
F = GF(q)


x, y = R.gens()

eval = vector(F, [1,2,3,4,5,6])
r = vector(F, [2,3,4,8,6,3])
n = 6

#floor(n*l/(l+1)-l/2*(k-1))
SudanListDecodingchat(eval, r, l, q, k,2)

AttributeError: type object 'R' has no attribute 'gens'

In [ ]:
def SudanListDecodingpaper(eval,r,l,q,k,s=1):
    n = eval.length()

    F = GF(q)
    R = PolynomialRing(F, "x,y")

    # make a univariate ring S = F[x] for the Popov matrix
    F = R.base_ring()
    S = PolynomialRing(F, "x")
    xS = S.gen()

    # keep bivariate variables only for final Q and factorization
    x, y = R.gens()

    # compute G and RX in S, not in R = F[x,y]
    G = prod(xS - S(xi) for xi in eval)
    L = LagrangePolynomials(eval, S, n)
    RX = R_Poly(L, r, n)

    # Compute the matrix BU
    BU = zero_matrix(S, l+1, l+1)

    for i in range(l+1):
        for a in range(l+1):
            if a > i:
                BU[a,i] = 0
            if i < s:
                BU[a,i] = binomial(i,a) * (-RX)**(i-a) * G**(s-i)
            else:
                if a == i:
                    BU[a,i] = 1
                else:
                    BU[a,i] = -sum((-RX)**(h-a)*binomial(h,a)*binomial(i,h)*(RX**(i-h)%G**(s-h)) for h in range(a,s))
                    print(M[a,i])

    # weighted degree should be x-degree + j*(k-1)
    W = vector(ZZ, [j*(k-1) for j in range(l+1)])

    BU = PopovForm(BU.transpose(),W)

    # Choose row with lowest weighted degree
    best_row = None
    best_degree = None
    for i in range(l+1):
        rowdegree = None
        for j in range(l+1):
            if M[i,j] == 0:       # skip zero entries
                continue

            deg = M[i,j].degree() + W[j]

            if rowdegree is None:
                rowdegree = deg
            else:
                rowdegree = max(deg, rowdegree)

        if rowdegree is not None and (best_degree is None or best_degree >= rowdegree):
            best_row = i
            best_degree = rowdegree

    # reconstruct Q in the bivariate ring R
    Q = 0
    for i in range(l+1):
        Q += R(M[best_row][i]) * y**i

    print(Q)

    candidates = []
    valid_candidates = []

    # multiplicity decoding condition
    threshold = floor(best_degree/s) + 1

    # Get fectors of Q and filter valid candidates
    for factor, _ in Q.factor():
        if factor.degree(y) == 1:
            a = factor.coefficient(y)
            b = factor - a*y

            f = -b / a
            candidates.append(f)

            agreements = 0
            for xi, ri in zip(eval, r):
                if f(xi, y) == ri:
                    agreements += 1

            if agreements >= threshold:
                valid_candidates.append(f)

    return candidates, valid_candidates

k = 2
l = 2
q = 13
F = GF(q)
eval = vector(F, [1,2,3,4])
r = vector(F, [2,3,4,8])

SudanListDecodingpaper(eval, r, l, q, k,1)

x + 9
11
0
1
-x^2 + y^2 - 2*x - 1


([-x - 1, x + 1], [x + 1])

In [ ]:
def is_popov_form(A,W=0):
    n, m = A.nrows(), A.ncols()

    # If no weights are given, set W = [0,0,...,0]
    if W == 0:
        W=zero_vector(ZZ, A.nrows())
        
    piv = [Pivot(A,i,W) for i in range(n)]  # (I,D)

    # gather nonzero rows
    nz = [i for i in range(n) if not A[i].is_zero()]

    # distinct pivot columns among nonzero rows
    cols = [piv[i][0] for i in nz]
    if len(set(cols)) != len(cols):
        return False

    # pivot-column reducedness: other entries in pivot col have strictly smaller degree
    for i in nz:
        I, D, DW = piv[i]
        for r in range(n):
            if r == i:
                continue
            v = A[r, I]
            deg = v.degree() if v != 0 else -1
            if deg >= D:
                return False

    return True

In [ ]:
F = GF(7)
R = PolynomialRing(F, "x")
x = R.gen()

A1 = Matrix(R, [
    [2*x**2 + 6*x + 1,   5*x**2 + x + 3,     4*x**2 + 6],
    [6*x + 4,          6*x + 3,           2*x + 1],
    [3*x**2 + x + 5,    3*x**2 + 2*x,        x**2 + 4],
])

W = [1,2,3]
Pf = PopovForm(A1,W)
print(Pf)
print("Popov?", is_popov_form(Pf,W))

[      4*x^2 + 4*x + 5                 x + 6                     0]
[        x^2 + 4*x + 4                     6                     1]
[x^3 + 3*x^2 + 5*x + 6                     1                     0]
Popov? True


In [ ]:
# Pivot tie inside a row (rightmost tie-break)
A2 = Matrix(R, [
    [x**2 + 1,      3*x**2 + 2,     5],      # tie deg 2 → pivot should be col 1
    [2*x + 4,      0,             x + 1],  # tie deg 1 → pivot should be col 2
    [0,            4*x + 6,       1],
])

Pf = PopovForm(A2)
print(Pf)
print("Popov?", is_popov_form(Pf))

[            0         x + 5             2]
[      2*x + 4             0         x + 1]
[x^2 + 5*x + 4             0             6]
Popov? True


In [ ]:
# Two rows start with the same pivot column (WeakPopov must fix)
A3 = Matrix(R, [
    [x**2 + 3*x + 1,    2*x + 1,      0],
    [3*x**2 + x + 4,    x + 6,        5],   # same initial pivot col as row 0
    [x + 2,            0,            1],
])

Pf = PopovForm(A3)
print(Pf)
print("Popov?", is_popov_form(Pf))

[x + 2     0     1]
[    5 x + 5     3]
[    4     2     x]
Popov? True


In [ ]:
# Same pivot column AND same pivot degree (hard collision)
A4 = Matrix(R, [
    [2*x**2 + 1,     4*x + 3,    6],
    [5*x**2 + 4,     x + 1,      2],   # same pivot col (0) and same deg (2)
    [x + 6,         3,          x],
])

Pf = PopovForm(A4)
print(Pf)
print("Popov?", is_popov_form(Pf))

[      1   x + 5       3]
[  x + 6       3       x]
[x^2 + 2       2       4]
Popov? True


In [ ]:
# Rank-deficient (a row becomes zero after reductions)
A5 = Matrix(R, [
    [x**2 + 2*x + 1,   3*x + 2,      4],
    [2*x**2 + 4*x + 2, 6*x + 4,      1],  # 2 * row0 in col0/col1; may cancel heavily
    [x + 3,           0,            5],
])

Pf = PopovForm(A5)
print(Pf)
print("Popov?", is_popov_form(Pf))

[      0       0       0]
[  x + 3       0       5]
[      2 5*x + 1   x + 1]
Popov? True


In [ ]:
# “Reduction can disturb other columns” stress (important for your PopovForm pass)
A6 = Matrix(R, [
    [x**2 + 1,      x**2 + 2,      0],
    [x**2 + 3,      0,            x**2 + 4],
    [0,            x + 1,        1],
])

Pf = PopovForm(A6)
print(Pf)
print("Popov?", is_popov_form(Pf))

[          0       x + 1           1]
[    x^2 + 1           3     6*x + 1]
[          2           4 x^2 + x + 3]
Popov? True


In [ ]:
# Already Popov-like sanity check (should pass quickly)
A7 = Matrix(R, [
    [x**2 + 1,     0,          0],
    [0,           x + 3,      0],
    [0,           0,          2*x + 1],
])

Pf = PopovForm(A7)
print(Pf)
print("Popov?", is_popov_form(Pf))

[      0   x + 3       0]
[      0       0   x + 4]
[x^2 + 1       0       0]
Popov? True


In [ ]:
# Rectangular (2x4): Popov should still reduce properly
A8 = Matrix(R, [
    [x**3 + 2*x + 1,   3*x**2 + 6,    x + 4,     2],
    [4*x**2 + x,       5*x**2 + 3*x,  6,         x + 1],
])

Pf = PopovForm(A8)
print(Pf)
print("Popov?", is_popov_form(Pf))

[    5*x^2 + 3*x       x^2 + 2*x               4         3*x + 3]
[x^3 + 6*x^2 + 1           x + 6           x + 6             5*x]
Popov? True


In [ ]:
# More rows than columns (4x3): forces zero row(s) in Popov output
A9 = Matrix(R, [
    [x**2 + 1,    x + 1,      1],
    [2*x**2 + 3,  2*x + 2,    2],
    [x + 4,      0,          x],
    [3*x**2 + 2,  x + 6,      4],
])

Pf = PopovForm(A9)
print(Pf)
print("Popov?", is_popov_form(Pf))

[0 0 0]
[1 0 0]
[0 1 0]
[0 0 1]
Popov? True


In [ ]:
import time
import statistics as stats
import random as pyrandom


def random_poly(R, x, max_deg, density=0.9):
    if pyrandom.random() > density:
        return R.zero()
    d = ZZ.random_element(0, max_deg + 1)
    c = [R.base_ring().random_element() for _ in range(d + 1)]
    while d > 0 and c[d] == 0:
        c[d] = R.base_ring().random_element()
    return sum(c[k] * x**k for k in range(d + 1))

def random_poly_matrix(R, x, n, m, max_deg, density=0.9):
    return Matrix(R, n, m, [random_poly(R, x, max_deg, density) for _ in range(n * m)])

def time_func(f, A, repeats=3):
    ts = []
    for _ in range(repeats):
        B = A.__copy__()
        t0 = time.perf_counter()
        f(B)
        ts.append(time.perf_counter() - t0)
    return ts

def run_benchmarks(
    sizes=((30,30),(50,50),(80,80),(120,120)),
    max_degs=(3,5,8),
    density=0.9,
    repeats=3,
    seed=0,
    warmup=True
):
    set_random_seed(seed)
    pyrandom.seed(seed)

    F = GF(7)
    R = PolynomialRing(F, "x")
    x = R.gen()

    if warmup:
        A0 = random_poly_matrix(R, x, 10, 10, 2, density=min(density, 0.7))
        WeakPopov(A0.__copy__())
        PopovForm(A0.__copy__())

    rows = []
    for (n, m) in sizes:
        for d in max_degs:
            A = random_poly_matrix(R, x, n, m, d, density=density)

            t_pf = time_func(PopovForm, A, repeats=repeats)

            rows.append({
                "n": n, "m": m, "max_deg": d, "density": density, "repeats": repeats,
                "PopovForm_mean_s": stats.mean(t_pf),
                "PopovForm_med_s": stats.median(t_pf),
                "PopovForm_min_s": min(t_pf),
            })

            print(f"(n,m)=({n},{m}) deg<={d} dens={density} | "
                  f"PopovForm mean {stats.mean(t_pf):.4f}s")

    return rows

results = run_benchmarks(
    sizes=((50,50),(80,80),(120,120)),
    max_degs=(3,5,8),
    density=0.85,
    repeats=3,
    seed=42
)

results[:2]

(n,m)=(50,50) deg<=3 dens=0.85 | PopovForm mean 17.0679s
(n,m)=(50,50) deg<=5 dens=0.85 | PopovForm mean 10.7689s
(n,m)=(50,50) deg<=8 dens=0.85 | PopovForm mean 9.1914s
(n,m)=(80,80) deg<=3 dens=0.85 | PopovForm mean 115.3277s
(n,m)=(80,80) deg<=5 dens=0.85 | PopovForm mean 98.8164s
(n,m)=(80,80) deg<=8 dens=0.85 | PopovForm mean 71.6138s
(n,m)=(120,120) deg<=3 dens=0.85 | PopovForm mean 577.8069s
(n,m)=(120,120) deg<=5 dens=0.85 | PopovForm mean 473.1777s
(n,m)=(120,120) deg<=8 dens=0.85 | PopovForm mean 402.3323s


[{'n': 50,
  'm': 50,
  'max_deg': 3,
  'density': 0.85,
  'repeats': 3,
  'PopovForm_mean_s': 17.067891610679606,
  'PopovForm_med_s': 16.876074303028872,
  'PopovForm_min_s': 16.59073820800404},
 {'n': 50,
  'm': 50,
  'max_deg': 5,
  'density': 0.85,
  'repeats': 3,
  'PopovForm_mean_s': 10.768926205666503,
  'PopovForm_med_s': 10.774103053001454,
  'PopovForm_min_s': 10.736955211992608}]

In [ ]:
import time
import statistics as stats
import random as pyrandom


def random_poly(R, x, max_deg, density=0.9):
    if pyrandom.random() > density:
        return R.zero()
    d = ZZ.random_element(0, max_deg + 1)
    c = [R.base_ring().random_element() for _ in range(d + 1)]
    while d > 0 and c[d] == 0:
        c[d] = R.base_ring().random_element()
    return sum(c[k] * x**k for k in range(d + 1))


def random_poly_matrix(R, x, n, m, max_deg, density=0.9):
    return Matrix(R, n, m,
                  [random_poly(R, x, max_deg, density) for _ in range(n * m)])


def time_func(f, A, repeats=3):
    ts = []
    for _ in range(repeats):
        B = A.__copy__()
        t0 = time.perf_counter()
        f(B)
        ts.append(time.perf_counter() - t0)
    return ts


def run_benchmarks(
    sizes=((30,30),(50,50),(80,80),(120,120)),
    max_degs=(3,5,8),
    density=0.9,
    repeats=3,
    seed=0,
    warmup=True
):
    set_random_seed(seed)
    pyrandom.seed(seed)

    F = GF(7)
    R = PolynomialRing(F, "x")
    x = R.gen()

    # Warmup Sage's popov_form
    if warmup:
        A0 = random_poly_matrix(R, x, 10, 10, 2, density=min(density, 0.7))
        A0.popov_form()

    rows = []
    for (n, m) in sizes:
        for d in max_degs:
            A = random_poly_matrix(R, x, n, m, d, density=density)

            t_pf = time_func(lambda B: B.popov_form(), A, repeats=repeats)

            rows.append({
                "n": n,
                "m": m,
                "max_deg": d,
                "density": density,
                "repeats": repeats,
                "popov_form_mean_s": stats.mean(t_pf),
                "popov_form_med_s": stats.median(t_pf),
                "popov_form_min_s": min(t_pf),
            })

            print(f"(n,m)=({n},{m}) deg<={d} dens={density} | "
                  f"popov_form mean {stats.mean(t_pf):.4f}s")

    return rows


results = run_benchmarks(
    sizes=((50,50),(80,80),(120,120)),
    max_degs=(3,5,8),
    density=0.85,
    repeats=3,
    seed=42
)

results[:2]


TypeError: The only supported seed types are:
None, int, float, str, bytes, and bytearray.

In [ ]:
import time
import statistics as stats
import random as pyrandom


def random_poly(R, x, max_deg, density=0.9):
    if pyrandom.random() > density:
        return R.zero()
    d = ZZ.random_element(0, max_deg + 1)
    c = [R.base_ring().random_element() for _ in range(d + 1)]
    while d > 0 and c[d] == 0:
        c[d] = R.base_ring().random_element()
    return sum(c[k] * x**k for k in range(d + 1))

def random_poly_matrix(R, x, n, m, max_deg, density=0.9):
    return Matrix(R, n, m, [random_poly(R, x, max_deg, density) for _ in range(n * m)])

def time_func(f, A, repeats=3):
    ts = []
    for _ in range(repeats):
        B = A.__copy__()
        t0 = time.perf_counter()
        f(B)
        ts.append(time.perf_counter() - t0)
    return ts

def run_benchmarks(
    sizes=((30,30),(50,50),(80,80),(120,120)),
    max_degs=(3,5,8),
    density=0.9,
    repeats=3,
    seed=0,
    warmup=True
):
    set_random_seed(seed)
    pyrandom.seed(seed)

    F = GF(7)
    R = PolynomialRing(F, "x")
    x = R.gen()

    if warmup:
        A0 = random_poly_matrix(R, x, 10, 10, 2, density=min(density, 0.7))
        PopovForm(A0.__copy__())

    rows = []
    for (n, m) in sizes:
        for d in max_degs:
            A = random_poly_matrix(R, x, n, m, d, density=density)

            t_pf = time_func(PopovForm, A, repeats=repeats)

            rows.append({
                "n": n, "m": m, "max_deg": d, "density": density, "repeats": repeats,
                "PopovForm_mean_s": stats.mean(t_pf),
                "PopovForm_med_s": stats.median(t_pf),
                "PopovForm_min_s": min(t_pf),
            })

            print(f"(n,m)=({n},{m}) deg<={d} dens={density} | "
                  f"PopovForm mean {stats.mean(t_pf):.4f}s")

    return rows

results = run_benchmarks(
    sizes=((50,50),(80,80),(120,120)),
    max_degs=(3,5,8),
    density=0.85,
    repeats=3,
    seed=42
)

results[:2]

(n,m)=(50,50) deg<=3 dens=0.85 | PopovForm mean 0.8098s
(n,m)=(50,50) deg<=5 dens=0.85 | PopovForm mean 0.6182s
(n,m)=(50,50) deg<=8 dens=0.85 | PopovForm mean 0.7025s
(n,m)=(80,80) deg<=3 dens=0.85 | PopovForm mean 4.9277s
(n,m)=(80,80) deg<=5 dens=0.85 | PopovForm mean 6.3275s
(n,m)=(80,80) deg<=8 dens=0.85 | PopovForm mean 2.7955s
(n,m)=(120,120) deg<=3 dens=0.85 | PopovForm mean 15.8018s
(n,m)=(120,120) deg<=5 dens=0.85 | PopovForm mean 16.6523s
(n,m)=(120,120) deg<=8 dens=0.85 | PopovForm mean 13.2168s


[{'n': 50,
  'm': 50,
  'max_deg': 3,
  'density': 0.85,
  'repeats': 3,
  'PopovForm_mean_s': 0.8098482843391442,
  'PopovForm_med_s': 0.7970187650062144,
  'PopovForm_min_s': 0.7787426660070196},
 {'n': 50,
  'm': 50,
  'max_deg': 5,
  'density': 0.85,
  'repeats': 3,
  'PopovForm_mean_s': 0.6181532943252629,
  'PopovForm_med_s': 0.6081634759902954,
  'PopovForm_min_s': 0.6022454009798821}]

In [ ]:
def ExtendedWeakPopov(A,V):
    n = A.nrows()
    m = A.ncols()+V.ncols()
    AV = zero_matrix(ZZ, n, m)
    for i in range(n):
        for j in range(m):
            if m<=A.ncols()
    pivots = zero_matrix(ZZ, n, 2)

    control = 1

    while control == 1:

        control = 0
        broke = False

        for i in range(n):
            pivots[i,0] = Pivot(A,i)[0]
            pivots[i,1] = Pivot(A,i)[1]

        for i in range(n):
            for j in range(n):
                if j != i:
                    if pivots[i,0] == pivots[j,0]:
                        if pivots[i,1] >= pivots[j,1]:
                            A = SimpleTransformation(A,i,j,pivots[i,0],pivots[i,1],pivots[j,1])
                        else:
                            A = SimpleTransformation(A,j,i,pivots[j,0],pivots[j,1],pivots[i,1])
                        control = 1
                        broke = True
                        break
            if broke:
                break
                        

    return A
    